# Exploring your taste

Before building a recommender, look at the data the way a data scientist would. This notebook belongs to the tutorial: [chapter 3](../docs/3-build-the-database.md) builds the database it reads, and [chapter 4](../docs/4-the-recommender.md) uses what it finds. It goes through eight steps:

1. Load and inspect the data
2. Handle missing values and duplicates
3. Standardise track identifiers and artist names
4. Clean numerical and categorical features
5. Scale numerical features
6. Encode genres
7. Visualise patterns in your music
8. Compare your favourite songs

Every step ends in a decision, and every number it rests on is computed below from your own data rather than typed in. Step 3's two decisions shaped the recommender: versions of a song are merged, and artists are told apart by their exact name.

**To run it:** build the database first (chapters 1 to 3), then open this notebook in VS Code or Jupyter, pick a Python 3.13 kernel with the requirements installed, and choose *Run All*; it takes under a minute. It opens `spotify.duckdb` read-only and only for a moment per query, so it can stay open; just don't run cells while a build in `scripts/` is running.

**Its outputs show your listening.** The repository keeps notebooks without outputs ([docs/privacy.md](../docs/privacy.md)), so clear them before you share a copy.

In [ ]:
import sys
from pathlib import Path

# The notebook runs in notebooks/, and the recommender package is in the project folder above it.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from recommender.data import SPOTIFY_DB, artist_key, letters, load_library, merge_songs, plain_titles
from recommender.features import genre_list, genre_matrix, genre_vocabulary
from recommender.profile import song_evidence


def sql(query: str) -> pd.DataFrame:
    """A query on spotify.duckdb, opened read-only for just this query."""
    with duckdb.connect(str(SPOTIFY_DB), read_only=True) as con:
        return con.sql(query).df()


library = load_library()
songs, plays = library.songs, library.plays
# Plays of music: tracks that config.toml marks as not music are left out here, as in the recommender.
music = plays[songs.is_music.to_numpy()[plays.song]]
print(f"{len(plays):,} plays of {len(songs):,} songs, {plays.started_at.min():%d %b %Y} to {plays.started_at.max():%d %b %Y}")

In [ ]:
# Chart style: the reference palette of the data-visualisation guidelines this project follows.
# Categorical colours are used in this fixed order, magnitudes in one blue ramp, and "Other" is gray.
SURFACE, INK, INK_2, MUTED, GRID, BASELINE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTHER = BASELINE
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 110,
    # Fallbacks for Japanese and Korean names.
    "font.family": ["Segoe UI", "Yu Gothic", "Malgun Gothic", "DejaVu Sans"], "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": INK, "axes.edgecolor": BASELINE, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": BASELINE, "ytick.color": BASELINE, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "legend.frameon": False, "lines.linewidth": 2, "lines.solid_capstyle": "round",
})


def thousands(value, _position=None) -> str:
    """Axis ticks as 0, 5K, 10K."""
    return f"{value / 1000:g}K" if abs(value) >= 1000 else f"{value:g}"


def no_frame(ax) -> None:
    """For heatmaps: no grid and no axis lines, the cells are the chart."""
    ax.grid(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(False)

## 1. Load and inspect the data

The recommender reads everything it needs at once into a `Library` (`recommender/data.py`): one row per song, one row per play, your likes and playlist adds, and the pools of new songs from Last.fm. Below: how big each part is, what a row looks like, and how your listening spreads over the years.

In [ ]:
pd.DataFrame({
    "rows": [len(songs), len(plays), len(library.feedback), len(library.candidates)],
    "one row per": ["song", "play", "like, unlike, playlist add or remove", "pool and song from Last.fm"],
}, index=["songs", "plays", "feedback", "candidates"])

In [ ]:
display(songs[["track_name", "artist_name", "album_name", "genres", "track_ids"]].head(3))
plays.head(3)

In [ ]:
per_year = plays.groupby(plays.started_at.dt.year).size()
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(per_year.index, per_year.to_numpy(), width=0.5, color=SERIES[0])
peak = per_year.idxmax()
ax.annotate(f"{per_year[peak]:,}", (peak, per_year[peak]), xytext=(0, 4), textcoords="offset points", ha="center", color=INK_2)
ax.set_title("Plays per year")
ax.set_xticks(per_year.index)
ax.yaxis.set_major_formatter(thousands)
ax.grid(axis="x", visible=False)
plt.show()
per_year.rename("plays").to_frame().T

## 2. Missing values and duplicates

**Missing values.** Spotify's exports have no genres, so they come from Last.fm (`scripts/fetch_genres.py`), which doesn't know every song. How much of a song you heard needs its length, which is estimated from your plays that ran to the end, so it's unknown for songs you never finished. Tracks that only appear in the technical log have no title or artist at all.

In [ ]:
has_genres = songs.genres.map(lambda g: len(genre_list(g)) > 0)
known_share = sql("SELECT avg(CAST(completion IS NOT NULL AS DOUBLE)) AS v FROM gold.plays").v[0]
log_only = sql("SELECT count(*) AS n FROM gold.tracks WHERE NOT is_played").n[0]
pd.DataFrame({"value": [
    f"{has_genres.mean():.1%}",
    f"{has_genres.to_numpy()[plays.song].mean():.1%}",
    f"{known_share:.1%}",
    f"{log_only:,}",
]}, index=[
    "songs with genres",
    "plays of songs with genres",
    "plays where the share heard is known",
    "tracks only in the technical log, without title or artist",
])

**What the recommender does about each.** A song without genres gets the median `genre_match` of the songs it's compared with, so missing data neither helps nor hurts it. A play whose share heard is unknown counts as half a listen if it ran at least 30 seconds, else 0 (`gold.plays.listen`). Tracks only seen in the technical log are left out: there is nothing to describe them by.

**Duplicates.** The export repeats some events word for word, and `build_database.py` keeps one copy. The bigger duplicate is one song under several track ids, which step 3 is about.

In [ ]:
counts = sql("""
    SELECT (SELECT count(*) FROM bronze.streams_raw) AS exported,
           (SELECT count(*) FROM silver.streams) AS without_repeats,
           (SELECT count(*) FROM silver.plays) AS music_plays
""").iloc[0]
pd.DataFrame({"rows": counts.to_numpy()}, index=[
    "streaming events, as exported",
    "after removing exact repeats",
    "plays of music (podcasts out, a play logged twice in the same second kept once)",
])

## 3. Standardise track identifiers and artist names

Spotify often has one song under several track ids: the single and the album version, a re-release, a language version, a remix. Until this phase the recommender merged tracks with exactly the same artist and title. Here that is compared with merging by **plain title**: the title without what Spotify adds to it, such as " - Remastered 2011", " (feat. …)", " - Japanese Ver.", " - Sped Up" or "(TV Size)". It is the same rule that already finds Last.fm songs in your history.

In [ ]:
tracks = sql("""
    SELECT t.track_id, t.track_name, t.artist_name, t.album_name, t.genres, t.is_music, t.est_duration_ms, s.plays
    FROM gold.tracks t JOIN gold.track_stats s USING (track_id)
    WHERE t.is_played
""")
tracks["artist_key"] = artist_key(tracks.artist_name)
tracks["exact"] = tracks.artist_key + "\n" + tracks.track_name.str.strip().str.casefold()
tracks["plain"] = tracks.artist_key + "\n" + plain_titles(tracks.track_name)
print(f"{len(tracks):,} played track ids")
print(f"{tracks.exact.nunique():,} songs when merged by exact title")
print(f"{tracks.plain.nunique():,} songs when merged by plain title: what recommender/data.py now does ({len(merge_songs(tracks)):,})")

In [ ]:
versions = tracks.groupby("plain").exact.nunique()
merged = tracks[tracks.plain.isin(versions[versions > 1].index)].sort_values(["plain", "plays"], ascending=[True, False])
examples = merged.groupby("plain").agg(
    artist=("artist_name", "first"),
    versions=("track_name", lambda titles: " | ".join(dict.fromkeys(titles))),
    plays=("plays", "sum"),
)
print(f"{len(examples)} songs gain versions this way, which brings together {merged.exact.nunique()} exact titles. The most played:")
examples.sort_values("plays", ascending=False).head(12).reset_index(drop=True)

Are those really the same song each time? A version that is much longer or shorter than the others would be a warning sign: a different song that happens to share the title.

In [ ]:
lengths = merged.groupby("plain").est_duration_ms.agg(["min", "max"]).dropna()
odd = lengths[lengths["max"] > 1.5 * lengths["min"]]
print(f"{len(odd)} of the {len(lengths)} merged songs with known lengths have a version over 1.5 times as long as another:")
merged[merged.plain.isin(odd.index)][["artist_name", "track_name", "est_duration_ms", "plays"]]

**Decision: merge versions.** They are the same songs: language versions, remixes, sped-up and acoustic versions, a TV edit. Kept apart, a version of a song you know can be offered to you as new, and your history with the song is split between its versions. `recommender/data.py` (`merge_songs`) now merges by plain title. The most played version gives the song its track id and title.

**Artist names.** First check: names that differ only in letter case, spaces or punctuation.

In [ ]:
tracks["letters"] = tracks.artist_name.map(letters)
spelt = tracks.groupby("letters").artist_name.nunique()
variants = tracks[tracks.letters.isin(spelt[spelt > 1].index)]
print("Names that differ only in letter case, spaces or punctuation, with their plays and one of their songs:")
variants.groupby(["letters", "artist_name"]).agg(plays=("plays", "sum"), song=("track_name", "first")).droplevel(0)

Are these one artist or two? Often they are **different artists** whose names differ only in letter case: in the author's history, two singers from different countries, and a group and a band. If the recommender ignored letter case, it would take each pair for one artist.

**Decision: tell artists apart by their exact name.** Spotify spells each artist one way, so case is part of the name; `artist_key()` in `recommender/data.py` keeps it. Last.fm, which ignores case, is matched by `artist_aliases()`. Names that differ only by punctuation are rarer and stay apart too: check the table above for yours.

The second check: different names with a song of the same title and length, which could be one artist under two names.

In [ ]:
known = tracks.dropna(subset=["est_duration_ms"]).assign(title=lambda t: plain_titles(t.track_name))
pairs = known.merge(known, on="title")
pairs = pairs[
    (pairs.artist_key_x < pairs.artist_key_y)
    & ((pairs.est_duration_ms_x - pairs.est_duration_ms_y).abs() <= 3000)
    & (pairs.title.str.len() > 3)
].drop_duplicates(["title", "artist_key_x", "artist_key_y"])
print(f"{len(pairs)} pairs of artist names with a song of the same title and length (within 3 seconds), for example:")
pairs[["title", "artist_name_x", "artist_name_y"]].head(12).reset_index(drop=True)

**Decision: leave these as they are.** They are mostly covers and remixes by other people, with the odd real rename among them, such as a band that changed its name. No rule tells those apart safely, and they are usually a small part of a history.

## 4. Clean numerical and categorical features

`ms_played`, the milliseconds of a play you heard, is the raw number behind every listening signal, and it's messy: seeking and replaying make a play longer than the song, and stopping a song in its first seconds is not the same as skipping it near the end. `build_gold.py` turns it, and the categorical `reason_end`, into three cleaner columns of `gold.plays`:

- `completion`: ms_played over the song's estimated length (the median of its plays that ran to the end), capped at 1;
- `end_type`: finished, skipped (you moved on) or interrupted (the app closed, the connection dropped), which says nothing about the song;
- `listen`: 1 if finished, 0 if skipped within 30 seconds, otherwise the completion. This is the number the recommender learns from.

In [ ]:
ends = sql("SELECT reason_end, end_type, count(*) AS plays FROM gold.plays GROUP BY ALL ORDER BY plays DESC")
ends["share"] = (ends.plays / ends.plays.sum()).map("{:.1%}".format)
over = sql("""
    SELECT avg(CAST(p.ms_played > 1.5 * t.est_duration_ms AS DOUBLE)) AS share
    FROM gold.plays p JOIN gold.tracks t USING (track_id)
    WHERE t.est_duration_ms > 0
""").share[0]
print(f"Plays more than 1.5 times as long as the song (seeking, replaying), which completion caps at 1: {over:.1%}")
ends.head(10)

In [ ]:
heard = sql("SELECT completion, end_type FROM gold.plays WHERE completion IS NOT NULL")
order = ["finished", "skipped", "interrupted"]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist([heard.completion[heard.end_type == end] for end in order], bins=np.linspace(0, 1, 21), stacked=True,
        color=SERIES[:3], label=order, edgecolor=SURFACE, linewidth=1.5)
ax.set_title("How much of the song you heard, by how the play ended")
ax.set_xlabel("share of the song heard (completion)")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.yaxis.set_major_formatter(thousands)
ax.legend()
ax.grid(axis="x", visible=False)
plt.show()
pd.crosstab(pd.cut(heard.completion, [0, 0.1, 0.5, 0.9, 1], include_lowest=True), heard.end_type)[order]

## 5. Scale numerical features

There are no audio features to scale: Spotify closed its audio-features API to new apps in November 2024. The numbers that do need scaling are the features of the learned rediscover model (`recommender/ranker.py`), for two reasons:

1. **Logs for long tails.** A few songs have hundreds of listens and most have a handful. A log turns "300 against 3" into a difference a linear model can weigh sensibly, which is why the model uses `log_listens`, `log_days_dormant` and `log_days_known`.
2. **Standardising.** Logistic regression's penalty on large weights treats every feature alike only when they are on the same scale, so each feature is shifted to mean 0 and divided by its standard deviation (`StandardScaler`). That is also what makes the learned weights comparable with each other.

In [ ]:
from sklearn.preprocessing import StandardScaler

from recommender.ranker import FEATURES, candidate_features
from recommender.scoring import rediscovery_candidates

cutoff = library.split_starts["test"]
listens = rediscovery_candidates(library, cutoff).listens
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
axes[0].hist(listens, bins=30, color=SERIES[0], edgecolor=SURFACE, linewidth=1)
axes[0].set_title("Listens per song")
axes[1].hist(np.log1p(listens), bins=30, color=SERIES[0], edgecolor=SURFACE, linewidth=1)
axes[1].set_title("The same, as log(1 + listens)")
axes[0].set_ylabel(f"songs to rediscover, {cutoff:%d %b %Y}")
for ax in axes:
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
features = candidate_features(library, cutoff)[list(FEATURES)]
scaled = StandardScaler().fit_transform(features)
pd.DataFrame({
    "mean": features.mean(), "standard deviation": features.std(ddof=0),
    "mean, scaled": scaled.mean(axis=0), "standard deviation, scaled": scaled.std(axis=0),
}).round(2)

## 6. Encode genres

A song can have several genres, so one-hot encoding (one column set to 1) doesn't fit. The recommender uses a **weighted multi-hot** encoding (`recommender/features.py`): one column per genre in your library, with 1 for the song's main genre, 1/2 for its second, 1/3 for its third, and each row then scaled to length 1, so that the cosine similarity of two songs is the dot product of their rows. Weighting rare genres up (TF-IDF) was tried in phase 1 and didn't rank better.

In [ ]:
vocabulary = genre_vocabulary(songs.genres)
rows = genre_matrix(songs.genres.tolist(), vocabulary)
genre_names = np.array(list(vocabulary))
print(f"{rows.shape[0]:,} songs x {rows.shape[1]} genres: {rows.nnz:,} cells are not zero ({rows.nnz / np.prod(rows.shape):.2%}), "
      f"{rows.nnz / has_genres.sum():.1f} genres per song that has any")
example = songs.index[songs.genres.map(lambda g: len(genre_list(g)) >= 3)][0]  # your most played song with 3+ genres
row = rows[example].toarray().ravel()
print(f"The row of {songs.track_name[example]} by {songs.artist_name[example]}:",
      ", ".join(f"{genre_names[i]} {row[i]:.3f}" for i in np.flatnonzero(row)))

In [ ]:
main_genre = songs.genres.map(lambda g: genre_list(g)[0] if genre_list(g) else None)
common = main_genre[songs.is_music].value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(common.index, common.to_numpy(), height=0.55, color=SERIES[0])
ax.annotate(f"{common.iloc[-1]:,}", (common.iloc[-1], len(common) - 1), xytext=(4, 0), textcoords="offset points",
            va="center", color=INK_2)
ax.set_title("The 15 most common main genres of your songs")
ax.set_xlabel("songs")
ax.grid(axis="y", visible=False)
plt.show()
common.sort_values(ascending=False).rename("songs").to_frame().T

## 7. Visualise patterns in your music

Four views of the same plays: what you listened to each year, when in the week you listen, which artists had their phase when, and how your plays spread over your songs.

In [ ]:
main_of_play = main_genre.fillna("no genre").to_numpy()[music.song]
year = music.started_at.dt.year.to_numpy()
share = pd.crosstab(year, main_of_play, normalize="index", rownames=["year"], colnames=["main genre"])
top = share.drop(columns="no genre", errors="ignore").mean().sort_values(ascending=False).head(7).index
shown = share[top].assign(Other=1 - share[top].sum(axis=1))
fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(shown))
for color, genre in zip(SERIES[:7] + [OTHER], shown.columns):
    ax.bar(shown.index, shown[genre], bottom=bottom, width=0.6, color=color, label=genre, edgecolor=SURFACE, linewidth=1.5)
    bottom += shown[genre].to_numpy()
ax.set_title("Your music each year by main genre, as a share of the year's plays")
ax.set_xticks(shown.index)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), reverse=True)
plt.tight_layout()
plt.show()
(shown * 100).round().astype(int).T.rename_axis("% of the year's plays")

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
# Local time in your time zone (config.toml), as build_database.py worked it out: 1 is Monday.
when = sql("SELECT local_dow, local_hour, count(*) AS plays FROM gold.plays WHERE is_music GROUP BY ALL")
when = when.pivot(index="local_dow", columns="local_hour", values="plays")
when = when.reindex(index=range(1, 8), columns=range(24)).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(10, 3.2))
image = ax.imshow(when.to_numpy(), cmap=BLUES, aspect="auto")
ax.set_yticks(range(7), days)
ax.set_xticks(range(0, 24, 2), [f"{hour:02d}" for hour in range(0, 24, 2)])
ax.set_xlabel("hour of the day (your time zone)")
ax.set_title("When you listen: plays by day and hour")
no_frame(ax)
fig.colorbar(image, ax=ax, label="plays", fraction=0.025, pad=0.01)
plt.show()
day, hour = np.unravel_index(when.to_numpy().argmax(), when.shape)
print(f"Busiest hour of the week: {days[day]} {hour:02d}:00 to {hour + 1:02d}:00, {when.to_numpy().max():,} plays over the years")
pd.DataFrame({"plays": when.sum(axis=1).to_numpy()}, index=days).T

In [ ]:
artist_of_play = songs.artist_name.to_numpy()[music.song]
by_artist = pd.crosstab(artist_of_play, year, rownames=["artist"], colnames=["year"])
top_artists = by_artist.sum(axis=1).sort_values(ascending=False).head(12).index
phase = by_artist.loc[top_artists] / by_artist.sum(axis=0)
fig, ax = plt.subplots(figsize=(9, 4.6))
image = ax.imshow(phase.to_numpy(), cmap=BLUES, aspect="auto")
ax.set_yticks(range(len(phase)), phase.index)
ax.set_xticks(range(len(phase.columns)), phase.columns)
no_frame(ax)
darkest = phase.to_numpy().max()
for row_number, peak_year in enumerate(phase.to_numpy().argmax(axis=1)):
    value = phase.iloc[row_number, peak_year]
    ax.text(peak_year, row_number, f"{value:.0%}", ha="center", va="center", fontsize=8,
            color="white" if value > darkest / 2 else INK)
fig.colorbar(image, ax=ax, label="share of that year's plays", fraction=0.03, pad=0.01, format=lambda v, _: f"{v:.0%}")
ax.set_title("Your 12 most played artists: their share of each year's plays, peak year labelled")
plt.show()
(phase * 100).round(1)

In [ ]:
per_song = music.groupby("song").size().sort_values(ascending=False)
cumulative = per_song.cumsum().to_numpy() / per_song.sum()
share_of_songs = np.arange(1, len(per_song) + 1) / len(per_song)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(share_of_songs * 100, cumulative * 100, color=SERIES[0])
for target in (0.5, 0.8):
    i = int(np.searchsorted(cumulative, target))
    ax.plot(share_of_songs[i] * 100, cumulative[i] * 100, "o", color=SERIES[0], markersize=8, markeredgecolor=SURFACE,
            markeredgewidth=2)
    ax.annotate(f"{share_of_songs[i]:.0%} of your songs take {target:.0%} of your plays", (share_of_songs[i] * 100, cumulative[i] * 100),
                xytext=(10, -6), textcoords="offset points", va="top", color=INK_2)
ax.set_title("A few songs take most of your plays")
ax.set_xlabel("songs, most played first (% of all)")
ax.set_ylabel("plays (cumulative %)")
plt.show()
print(f"{len(per_song):,} songs of music; the most played has {per_song.iloc[0]:,} plays, and half have {per_song.median():.0f} or fewer")

## 8. Compare your favourite songs

Here your favourites are the 150 songs of music with the highest weight in your taste profile (`recommender/profile.py`): played a lot, and lately. What do they have in common, and how do they differ from each other?

In [ ]:
evidence = song_evidence(library, library.ends_at)
weight = evidence.weight.where(songs.is_music.to_numpy())
favourites = weight.sort_values(ascending=False).head(150).index.to_numpy()
songs.loc[favourites[:10], ["track_name", "artist_name"]].assign(plays=evidence.plays.loc[favourites[:10]].astype(int))

In [ ]:
def genre_share(song_numbers) -> pd.Series:
    """Share of the songs that have each genre, at any rank."""
    return songs.genres.iloc[song_numbers].map(genre_list).explode().dropna().value_counts() / len(song_numbers)


all_music = np.flatnonzero(songs.is_music.to_numpy())
compare = pd.DataFrame({"all your songs": genre_share(all_music), "favourites": genre_share(favourites)}).fillna(0)
compare = compare.sort_values("favourites", ascending=False).head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
y = np.arange(len(compare))
ax.hlines(y, compare["all your songs"], compare["favourites"], color=BASELINE, linewidth=2)
ax.plot(compare["all your songs"], y, "o", color="#86b6ef", markersize=8, markeredgecolor=SURFACE, markeredgewidth=2,
        label="all your songs")
ax.plot(compare["favourites"], y, "o", color="#1c5cab", markersize=8, markeredgecolor=SURFACE, markeredgewidth=2,
        label="your 150 favourites")
ax.set_yticks(y, compare.index)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Genres of your favourites against all your songs (share of songs)")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
plt.show()
(compare.iloc[::-1] * 100).round(1)

In [ ]:
first_played = plays.groupby("song").started_at.min()


def behaviour(song_numbers) -> pd.Series:
    rows_ = evidence.loc[song_numbers]
    return pd.Series({
        "songs": len(song_numbers),
        "median plays": rows_.plays.median(),
        "share of plays finished": rows_.finished.sum() / rows_.plays.sum(),
        "share of plays skipped within 30 seconds": rows_.early_skips.sum() / rows_.plays.sum(),
        "median years since the first play": ((library.ends_at - first_played.loc[song_numbers]).dt.days / 365.25).median(),
    })


pd.DataFrame({"favourites": behaviour(favourites), "all your songs": behaviour(all_music)}).round(2)

**How your favourites differ from each other.** k-means clustering puts songs into k groups so that each song is closest to its group's average genre row. The silhouette says how well the groups hold together: near 1, every song sits much closer to its own group than to the next; near 0, the groups blur into each other. It is computed with cosine distance, the same similarity the recommender uses. A clear number of taste groups would show as a peak in the silhouette.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

clustered = favourites[has_genres.to_numpy()[favourites]]
X = rows[clustered].toarray()
silhouette = {
    k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X), metric="cosine") for k in range(2, 11)
}
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(list(silhouette), list(silhouette.values()), color=SERIES[0], marker="o", markersize=8, markeredgecolor=SURFACE,
        markeredgewidth=2)
ax.set_title("How cleanly your favourites split into k groups by genre")
ax.set_xlabel("number of groups, k")
ax.set_ylabel("silhouette")
ax.set_xticks(list(silhouette))
plt.show()
display(pd.Series(silhouette, name="silhouette").round(3).to_frame().T)
genre_lists = songs.genres.iloc[clustered].map(lambda g: "|".join(genre_list(g)))
print(f"{len(clustered)} favourites with genres have {genre_lists.nunique()} different genre lists; the most common:")
genre_lists.value_counts().head(5).rename("favourites").to_frame()

**There is no peak.** Every group added scores better, because many favourites have exactly the same genres: songs tagged with the same single genre are one point, and a point is always a perfect group. So genres alone don't say how many tastes you have. Below, your favourites in six groups, a number picked for reading, not one the data asked for.

In [ ]:
k = 6
model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
groups = pd.Series(model.labels_, index=clustered)
summary = []
for group in range(k):
    members = groups.index[groups == group]
    centre = model.cluster_centers_[group]
    summary.append({
        "songs": len(members),
        "main genres": ", ".join(genre_names[np.argsort(centre)[::-1][:3]]),
        "most common artists": ", ".join(songs.artist_name.loc[members].value_counts().head(3).index),
    })
pd.DataFrame(summary).sort_values("songs", ascending=False, ignore_index=True)

## 9. What this means for the recommender

Rerun the notebook after a newer export and every number above updates. What to look for, and what the recommender does about it:

**Decisions this notebook led to**, all in the code now:

- **Versions of one song are merged by plain title** (step 3). Check that the merged songs' versions run about as long as each other; in the author's history only a TV edit stood out. Kept apart, a version of a song you know could be offered to you as new.
- **Artists are told apart by their exact name** (step 3), letter case included. Covers, renames and punctuation variants are left as they are.
- **Missing genres get a neutral match** (step 2): a song without genres gets the median genre match of the songs it is compared with, and a play of unknown length counts as half a listen if it ran 30 seconds.
- **Long-tailed features are logged and standardised** (step 5), and genres are weighted multi-hot rows (step 6).

**Patterns that shape the recommender**, which steps 7 and 8 show for your data:

- **Taste moves in phases.** When an artist takes a big share of one year and little of the next, old plays should fade: the profile counts a play half as much for every 180 days of age. It is also why two test windows can disagree.
- **A few songs take most of the plays.** That long tail is why song weights are logged.
- **Plays either run to the end or stop early.** When most plays are finished or skipped in their first tenth, `listen` (1, 0 or the share heard) loses little by being simple.
- **Genres sort favourites only roughly.** When many favourites share the same few genre lists and the silhouette has no peak, genres alone can't tell your tastes apart. That is the case for learning which songs you play together from your sessions, as `recommender/sessions.py` does ([chapter 4](../docs/4-the-recommender.md)).